# NPE tuning

In [18]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import warnings
import sys
sys.path.append('../../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

## Load simulation data

Load genome data and clonal tree.

In [2]:
drop_col = range(16, 32)
theta_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")
x_test = np.delete(x_test, drop_col, axis=1)
print(theta_test.shape, x_test.shape)

nan_row_test = np.where(np.isnan(x_test) | np.isinf(x_test))[0]
print(nan_row_test)

theta_test = np.delete(theta_test, nan_row_test, axis=0)
theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, nan_row_test, axis=0)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

(1000, 3) (1000, 30)
[865 899]
torch.Size([998, 3]) torch.Size([998, 30])


In [3]:
theta1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")
theta2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

x = np.vstack([x1, x2])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1, theta2])
print(theta.shape, x.shape)

nan_row = np.where(np.isnan(x) | np.isinf(x))[0]
print(nan_row)

theta = np.delete(theta, nan_row, axis=0)
theta = torch.tensor(theta[:10000, :], device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = np.delete(x, nan_row, axis=0)
x = torch.tensor(x[:10000, :], device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

print(theta.shape, x.shape)

(20000, 3) (20000, 30)
[  114   681   706  1448  2554  2818  7211  7282  7329  7392  8938  9827
  9973 10223 10788 12192 13567 14388 14653]
torch.Size([10000, 3]) torch.Size([10000, 30])


## Test functions

In [4]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    print("Kolmogorov-Smirnov Test Results (SBC):")
    print("-" * 40)

    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)

        print(f"Parameter:        {parameter_labels[dim]}")
        print(f"KS Statistic (D): {ks_stat:.4f}")
        print(f"p-value:          {p_value:.4e}")

        if p_value < 0.05:
            print("Status: MISCALIBRATED (reject null)")
        else:
            print("Status: CALIBRATED (fail to reject null)")
        print("-" * 40)
    
    return ks_results, p_values

In [5]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## Tuning setting

In [8]:
seeds = [1, 2, 3, 4, 5]
num_posterior_samples=500

## Baseline NPE

In [9]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))
prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [10]:
embedding_net = LeaveLengthOut_NN(
    input_dim=30,
    num_hiddens=48,
    num_hidden_layers=2,
    num_outputs=4)
neural_posterior = posterior_nn(
    model="nsf", 
    embedding_net=embedding_net 
)

In [11]:
seed = seeds[0]
inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
torch.manual_seed(seed)
np.random.seed(seed)

In [12]:
density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
    max_num_epochs=500
)
posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

 Neural network successfully converged after 120 epochs.

In [13]:
theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
    theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
                                           show_progress_bars=False, reject_outside_prior=False)
    theta_est_post[j, :, :] = theta_post.detach().numpy()

Sampling posterior: 100%|██████████| 998/998 [00:14<00:00, 67.61it/s]


In [15]:
sbc_results = []
parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
theta_test_expanded = theta_test.unsqueeze(1)
theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
is_less_than_truth = theta_est_post_tensor < theta_test_expanded
ranks = torch.sum(is_less_than_truth, dim=1)

ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)

Kolmogorov-Smirnov Test Results (SBC):
----------------------------------------
Parameter:        for $\rho_s$
KS Statistic (D): 0.0450
p-value:          3.4070e-02
Status: MISCALIBRATED (reject null)
----------------------------------------
Parameter:        for $\theta_s$
KS Statistic (D): 0.0750
p-value:          2.4761e-05
Status: MISCALIBRATED (reject null)
----------------------------------------
Parameter:        for L
KS Statistic (D): 0.2278
p-value:          5.3790e-46
Status: MISCALIBRATED (reject null)
----------------------------------------


In [16]:
mahalanobis_error(theta_est_post, theta_test_numpy)

array([ 1.16224868,  0.37754256,  3.13548288,  1.24109227,  0.91847553,
        1.45941865,  0.6843219 ,  1.50476567,  0.28694883,  1.43020419,
        1.63908742,  1.50331615,  1.33075148,  1.75723481,  0.47530063,
        1.07355158,  1.06350468,  0.33819539,  0.98760148,  1.41503152,
        1.2051767 ,  1.398313  ,  1.04888808,  2.68547813,  3.17883799,
        0.93753059,  1.38059797,  0.24152472,  1.87369583,  1.2470283 ,
        1.30666616,  1.76556456,  0.97022976,  0.98707575,  0.95869973,
        0.71868697,  0.73103182,  1.7526865 ,  2.56828223,  1.36381604,
        0.32927922,  1.22745305,  1.82037031,  1.36129108,  0.52540165,
        0.94283622,  1.08337436,  0.65442437,  0.15174491,  0.17478798,
        1.42303506,  0.58527039,  1.4091982 ,  0.12799455,  3.16235793,
        0.61237509,  1.82177017,  1.43812064,  2.19392473,  0.48454443,
        0.74696127,  0.60697414,  1.57547155,  1.16403447,  1.28676622,
        1.8585495 ,  0.92452821,  0.25078119,  2.30914108,  1.29

In [29]:
def compute_validation_nll(density_estimator, theta_val, x_val, device="cpu", batch_size=None):
    """
    Compute validation negative log likelihood for a trained sbi NPE density estimator.

    For NPE, the density estimator approximates q(theta | x), so the validation
    NLL is:

        -mean(log q(theta_val | x_val))

    Lower validation NLL is better.

    Important: For sbi density estimators based on nflows, the conditioning
    variable should usually be passed as `context=x_val`, not `x=x_val`.

    Parameters
    ----------
    density_estimator : torch.nn.Module
        Trained density estimator returned by inference.train().
    theta_val : torch.Tensor
        Held-out validation parameters.
    x_val : torch.Tensor
        Held-out validation observations.
    device : str or torch.device
        Device used for evaluation.
    batch_size : int or None
        Optional mini-batch size for evaluation. Use None to evaluate all at once.

    Returns
    -------
    float
        Mean validation negative log likelihood.
    """
    density_estimator.eval()
    device = torch.device(device)

    theta_val = theta_val.to(device)
    x_val = x_val.to(device)

    log_probs = []

    with torch.no_grad():
        if batch_size is None:
            lp = density_estimator.log_prob(theta_val, x_val)
            log_probs.append(lp.detach().cpu())
        else:
            num_val = theta_val.shape[0]
            for start in range(0, num_val, batch_size):
                end = min(start + batch_size, num_val)
                lp = density_estimator.log_prob(
                    theta_val[start:end],
                    x_val[start:end],
                )
                log_probs.append(lp.detach().cpu())

    log_probs = torch.cat(log_probs, dim=0)
    validation_nll = -log_probs.mean().item()

    return validation_nll

In [30]:
compute_validation_nll(density_estimator_baseline, theta_test, x_test)

-4.495190143585205